## Setup

In [ ]:
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/data_utils.py

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import re

from sklearn.compose import make_column_transformer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder, PolynomialFeatures, StandardScaler
from sklearn.svm import SVC

from data_utils import object_from_json_url, classification_error, display_confusion_matrix

## `AwesomePipeline` class

Abstracts the details of setting up, training and predicting

In [ ]:
class AwesomePipeline():
  def __init__(self, outcome_cols, numerical_cols, categorical_cols, model, poly_degree=1):
    self.out_cols = outcome_cols
    self.num_cols = numerical_cols
    self.cat_cols = categorical_cols

    self.model = make_pipeline(
      make_column_transformer(
        (StandardScaler(), self.num_cols),
        (OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), self.cat_cols),
      ),
      PolynomialFeatures(degree=poly_degree, include_bias=False),
      model
    )

  def fit(self, train_data):
    if len(self.out_cols) > 0:
      self.model.fit(train_data.drop(columns=self.out_cols), train_data[self.out_cols])
    else:
      self.model.fit(train_data)

  def predict(self, data):
    return self.model.predict(data)

## Regression: Head Circumference

In [ ]:
ANSUR_FILE = "https://raw.githubusercontent.com/PSAM-5020-2026S-A/5020-utils/main/datasets/json/ansur.json"
ansur_data = object_from_json_url(ANSUR_FILE)
ansur_df = pd.json_normalize(ansur_data)
ansur_df

In [ ]:
train_df, test_df = train_test_split(ansur_df, test_size=0.2, random_state=1010)

num_cols = ansur_df.drop(columns=["head.circumference", "gender"]).columns

mPipeline = AwesomePipeline(
  outcome_cols = ["head.circumference"],
  numerical_cols=num_cols,
  categorical_cols=["gender"],
  model=LinearRegression(),
  poly_degree=1
)

mPipeline.fit(train_df)

train_pred = mPipeline.predict(train_df)
test_pred = mPipeline.predict(test_df)

print(
  root_mean_squared_error(train_df["head.circumference"], train_pred),
  root_mean_squared_error(test_df["head.circumference"], test_pred)
)

## Regression: Diamond Prices

In [ ]:
DIAMOND_FILE = "https://raw.githubusercontent.com/PSAM-5020-2026S-A/5020-utils/main/datasets/json/diamonds.json"
diamond_data = object_from_json_url(DIAMOND_FILE)
diamond_df = pd.json_normalize(diamond_data)
diamond_df

In [ ]:
train_df, test_df = train_test_split(diamond_df, test_size=0.2, random_state=1010)

mPipeline = AwesomePipeline(
  outcome_cols = ["price"],
  numerical_cols=["carat", "depth", "table", "x", "y", "z"],
  categorical_cols=["cut", "color", "clarity"],
  model=LinearRegression(),
  poly_degree=3
)

mPipeline.fit(train_df)

train_pred = mPipeline.predict(train_df)
test_pred = mPipeline.predict(test_df)

print(
  root_mean_squared_error(train_df["price"], train_pred),
  root_mean_squared_error(test_df["price"], test_pred)
)

## Classification: Wine Quality

In [ ]:
WINES_FILE = "https://raw.githubusercontent.com/PSAM-5020-2026S-A/5020-utils/main/datasets/json/wines.json"
wine_data = object_from_json_url(WINES_FILE)
wine_df = pd.json_normalize(wine_data)
wine_df

In [ ]:
train_df, test_df = train_test_split(wine_df, test_size=0.2, random_state=1010)

num_cols = [
  "acidity", "volatility", "citrics", "sugar", "chlorides",
  "sulfur dioxide", "density", "pH", "sulphates", "alcohol"
]

mPipeline = AwesomePipeline(
  outcome_cols = ["quality"],
  numerical_cols=num_cols,
  categorical_cols=[],
  model=SVC(),
  poly_degree=2
)

mPipeline.fit(train_df)

train_pred = mPipeline.predict(train_df)
test_pred = mPipeline.predict(test_df)

print(
  classification_error(train_df["quality"], train_pred),
  classification_error(test_df["quality"], test_pred)
)

## Classification: Penguins

In [ ]:
PENGUIN_URL = "https://raw.githubusercontent.com/PSAM-5020-2026S-A/5020-utils/refs/heads/main/datasets/json/penguins.json"
penguin_data = object_from_json_url(PENGUIN_URL)
penguin_df = pd.json_normalize(penguin_data)

# Manually encode output feature
penguin_labels = sorted(penguin_df["species"].unique().tolist())

def L2S(L): return penguin_labels.index(L)
penguin_df["species"] = penguin_df["species"].apply(L2S)

penguin_df

In [ ]:
train_df, test_df = train_test_split(penguin_df, test_size=0.2, random_state=1010)

num_cols = [
  "bill_length_mm", "bill_depth_mm",
  "flipper_length_mm", "body_mass_g"
]

mPipeline = AwesomePipeline(
  outcome_cols = ["sex", "species"],
  numerical_cols=num_cols,
  categorical_cols=[],
  model=RandomForestClassifier(),
  poly_degree=1
)

mPipeline.fit(train_df)

train_pred = pd.DataFrame(mPipeline.predict(train_df), columns=["sex", "species"])
test_pred = pd.DataFrame(mPipeline.predict(test_df), columns=["sex", "species"])

print(classification_error(train_df["sex"], train_pred["sex"]))
print(classification_error(train_df["species"], train_pred["species"]))

display_confusion_matrix(test_df["sex"], test_pred["sex"], ["F","M"])
display_confusion_matrix(test_df["species"], test_pred["species"], penguin_labels)

## Clustering: Headphones

In [ ]:
ANSUR_FILE = "https://raw.githubusercontent.com/PSAM-5020-2026S-A/5020-utils/main/datasets/json/ansur.json"
ansur_data = object_from_json_url(ANSUR_FILE)
ansur_df = pd.json_normalize(ansur_data)
ansur_df

In [ ]:
train_df, test_df = train_test_split(ansur_df, test_size=0.2, random_state=1010)

num_cols = [
  "height", "weight",
  "head.height", "head.circumference",
  "ear.breadth", "ear.length", "ear.protrusion"
]

mPipeline = AwesomePipeline(
  outcome_cols = [],
  numerical_cols=num_cols,
  categorical_cols=[],
  model=KMeans(n_clusters=4),
  poly_degree=1
)

mPipeline.fit(train_df)

train_df["cluster"] = mPipeline.predict(train_df)
test_df["cluster"] = mPipeline.predict(test_df)

display(train_df)
display(test_df)